# 04b: What did the model learn? (3 ML charts)

These three charts explain the model in plain words, for an audience with no medical or ML background:

| Chart | Question it answers |
|---|---|
| **R1** Top factors | Which information matters most, and does it make a serious report more or less likely? |
| **R2** The four drugs | Once side effects, reporter and age are taken into account, which drug's reports still look riskier? |
| **R3** Risk groups | If reviewers read the riskiest reports first, how many serious ones do they find? |

Model-performance charts (model comparison, confusion matrix, ROC/PR curves) are in Brian's notebook.

**How the model is built here:** the same setup the team agreed on: the model-ready file from `03a_ml_prep`,
a stratified 80/20 split with `random_state=42`, and class weights for the 11% imbalance.
The model settings are in one cell (Step 2). If the team's final model uses different settings, change them there
and re-run all cells: every chart and number updates.

**Explainability method:** SHAP. For every report, SHAP splits the model's prediction into a contribution from each factor,
like splitting a restaurant bill so you can see who ordered what.

In [ ]:
from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score, roc_auc_score
import xgboost as xgb
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = PROJECT_ROOT / "data" / "03_model" / "glp1_model_ready.parquet"
FIG_DIR = PROJECT_ROOT / "figures"
ML_DIR = PROJECT_ROOT / "outputs" / "ml"
FIG_DIR.mkdir(exist_ok=True)
ML_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Chart titles state the finding. Check them after re-running with new data or a new model.
R1_TITLE = "Pen, dose and injection-site complaints signal a mild report; a long side-effect list signals a serious one"
R2_TITLE = "Same side effects, same reporter: liraglutide and semaglutide reports still look riskier"

# Chart style: large text for slides, one highlight colour, everything else grey
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 14, "axes.titlesize": 19, "axes.titleweight": "bold", "axes.labelsize": 14,
    "xtick.labelsize": 13, "ytick.labelsize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
HIGHLIGHT = "#1F4E79"   # dark blue
RAISES = "#E07B39"      # orange = pushes towards serious
LOWERS = "#1F4E79"      # dark blue = pushes away from serious
GREY = "#B9BDC5"
SOURCE = "Source: FDA FAERS, GLP-1 reports 2022 Q1 - 2026 Q2 (252,568 reports). Test set = 20% of reports the model never saw."


def title(ax, text, width=70):
    ax.set_title("\n".join(textwrap.wrap(text, width)), loc="left", pad=14)


def caption(fig, text, y=-0.02):
    fig.text(0.01, y, "\n".join(textwrap.wrap(text, 150)), fontsize=11, color="#555555", ha="left", va="top")

## Step 1: Load the model-ready data and split it

The split keeps the 11% serious share the same in the training part (80%) and the test part (20%).
All charts below use the **test part only**: reports the model has never seen, like a final exam with new questions.

In [ ]:
df = pd.read_parquet(DATA)
ID_COLS, TARGET = ["primaryid", "caseid"], "is_serious"
X_raw = df.drop(columns=ID_COLS + [TARGET])
y = df[TARGET]

CAT_COLS = [c for c in X_raw.columns if not pd.api.types.is_numeric_dtype(X_raw[c])]
X = pd.get_dummies(X_raw, columns=CAT_COLS, prefix_sep="=", dtype=int)

X_train, X_test, y_train, y_test, raw_train, raw_test = train_test_split(
    X, y, X_raw, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
ids_test = df.loc[X_test.index, ID_COLS]

print(f"Train: {len(X_train):,} reports ({y_train.mean()*100:.1f}% serious)")
print(f"Test:  {len(X_test):,} reports ({y_test.mean()*100:.1f}% serious)")
print(f"Model columns after one-hot encoding: {X.shape[1]}")

## Step 2: Train the model

Gradient-boosted trees (XGBoost): many small decision trees, each one fixing the mistakes of the ones before.
`scale_pos_weight` tells the model that missing a serious report costs about 8 times more than a false alarm
(the ratio of non-serious to serious reports), so it does not just predict "not serious" every time.

In [ ]:
neg_pos = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=neg_pos, eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]
THRESHOLD = 0.5
pred = (prob >= THRESHOLD).astype(int)

metrics = pd.DataFrame([{
    "model": "XGBoost",
    "recall": recall_score(y_test, pred), "precision": precision_score(y_test, pred),
    "f1": f1_score(y_test, pred), "pr_auc": average_precision_score(y_test, prob),
    "roc_auc": roc_auc_score(y_test, prob), "threshold": THRESHOLD}]).round(3)
metrics

### Plain-English names for the model's columns

In [ ]:
dictionary_path = PROJECT_ROOT / "reports" / "ml_feature_dictionary.csv"
group_names = {}
if dictionary_path.exists():
    d = pd.read_csv(dictionary_path)
    for f, m in zip(d["feature"], d["meaning"].fillna("")):
        hit = re.search(r"'(.+)'", m)
        if f.startswith("rg_") and hit:
            group_names[f] = hit.group(1)

CAT_LABEL = {"drug": "Drug", "age_group": "Age", "sex": "Sex", "reporter_type": "Reported by",
             "indication_group": "Taken for", "route_group": "Given as"}
VALUE_LABEL = {"F": "female", "M": "male", "65 and over": "65+", "Injection": "injection", "Oral": "pill",
               "Health professional": "other health professional"}
NUM_LABEL = {"n_reactions": "Number of side effects listed", "n_other_drugs": "Number of other medicines listed",
             "dose_reported": "Dose given in the report", "year": "Year received"}


def nice(col):
    if "=" in col:
        base, val = col.split("=", 1)
        val = VALUE_LABEL.get(val, val)
        if base == "drug":
            return f"Drug: {val}"
        return f"{CAT_LABEL.get(base, base)}: {val if val != 'Unknown' else 'not stated'}"
    if col.startswith("rg_"):
        return "Side effect: " + group_names.get(col, col[3:].replace("_", " ")).lower()
    return NUM_LABEL.get(col, col)


print([nice(c) for c in X.columns[:12]])

## Step 3: SHAP values

SHAP is computed for a random sample of 20,000 test reports (enough for stable averages, and fast).
XGBoost calculates exact SHAP values itself (`pred_contribs=True`), so the separate `shap` package is not needed.

In [ ]:
sample = X_test.sample(n=min(20000, len(X_test)), random_state=RANDOM_STATE)
# XGBoost computes exact TreeSHAP values itself (pred_contribs=True); the last column is the baseline
contribs = model.get_booster().predict(xgb.DMatrix(sample), pred_contribs=True)
shap_df = pd.DataFrame(contribs[:, :-1], columns=sample.columns, index=sample.index)

importance = (shap_df.abs().mean().sort_values(ascending=False)
              .rename("mean_abs_shap").to_frame())
importance["feature"] = [nice(c) for c in importance.index]
importance.head(12)

## R1: Which information matters most, and which way does it push?

**Bar length** = how much the model leans on that information (share of all SHAP importance).
**Colour** = which way it pushes, for reports that have it: **orange** = towards "serious", **blue** = away from "serious".
For number columns (for example the number of side effects listed) the direction is for reports in the top quarter.

In [ ]:
def push_when_present(col):
    """Average SHAP value for reports that have the factor (top quarter for number columns)."""
    if set(np.unique(sample[col])) <= {0, 1}:
        mask = sample[col] == 1
    else:
        mask = sample[col] >= sample[col].quantile(0.75)
    return shap_df.loc[mask, col].mean() if mask.any() else 0.0

TOP = 10
top = importance.head(TOP).copy()
top["share"] = top["mean_abs_shap"] / importance["mean_abs_shap"].sum() * 100
top["push"] = [push_when_present(c) for c in top.index]
labels = []
for c, f in zip(top.index, top["feature"]):
    if c == "n_reactions" or c == "n_other_drugs":
        f = f + f" ({sample[c].quantile(0.75):g}+)"
    labels.append(f)
top["label"] = labels
top = top.iloc[::-1]

fig, ax = plt.subplots(figsize=(12, 6.75))
colors = [RAISES if p > 0 else LOWERS for p in top["push"]]
bars = ax.barh(top["label"], top["share"], color=colors, height=0.65)
for b, v in zip(bars, top["share"]):
    ax.text(b.get_width() + 0.25, b.get_y() + b.get_height() / 2, f"{v:.0f}%", va="center", fontsize=13)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=RAISES, label="Makes a serious report MORE likely"),
                   Patch(color=LOWERS, label="Makes a serious report LESS likely")],
          loc="lower right", frameon=False, fontsize=13)
ax.set_xlabel("Share of the model's decision-making (%)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlim(0, top["share"].max() * 1.15)
ax.grid(axis="x", alpha=0.25)
title(ax, R1_TITLE)
caption(fig, f"Top 10 of {X.shape[1]} model inputs, ranked by mean absolute SHAP value, shown as a share of the total. "
        "Colour = sign of the average SHAP value for reports that have the factor. Linked to, not caused by: "
        "these are reporting patterns, not proof that a factor causes harm. " + SOURCE)
fig.savefig(FIG_DIR / "R1_top_factors.png")
plt.show()
top.iloc[::-1][["label", "share", "push"]].round(3)

## R2: Which drug's reports look riskier, once everything else is taken into account?

The raw numbers (from the EDA) compare drugs **as reported**, but the drugs differ in who reports them and which side effects
are listed (for example, many tirzepatide reports are consumer reports about pens and doses).
Here SHAP isolates the **drug's own push** on the model's risk score, *after* side effects, reporter, age and the rest
are taken into account: like comparing restaurants after adjusting for what each customer ordered.

In [ ]:
DRUG_COLS = [c for c in sample.columns if c.startswith("drug=")]
drug_push = (pd.DataFrame({"drug": raw_test.loc[sample.index, "drug"],
                           "push": shap_df[DRUG_COLS].sum(axis=1)})
             .groupby("drug")["push"].mean())
raw_pct = df.groupby("drug")[TARGET].mean() * 100
drugs = drug_push.sort_values().index

BRAND = {"semaglutide": "Semaglutide\n(Ozempic, Wegovy, Rybelsus)", "tirzepatide": "Tirzepatide\n(Mounjaro, Zepbound)",
         "dulaglutide": "Dulaglutide\n(Trulicity)", "liraglutide": "Liraglutide\n(Victoza, Saxenda)"}

fig, ax = plt.subplots(figsize=(12, 6.75))
vals = drug_push[drugs]
ylabels = [f"{BRAND.get(d, d)}\n{raw_pct[d]:.0f}% of its reports serious" for d in drugs]
ax.barh(ylabels, vals, color=[RAISES if v > 0 else LOWERS for v in vals], height=0.6)
ax.axvline(0, color="#333333", lw=1)
lim = vals.abs().max() * 1.3
ax.set_xlim(-lim, lim)
ax.set_xticks([])
ax.text(lim * 0.98, -0.75, "Pushes risk UP  →", ha="right", va="center", color=RAISES, fontsize=14, fontweight="bold")
ax.text(-lim * 0.98, -0.75, "←  Pushes risk DOWN", ha="left", va="center", color=LOWERS, fontsize=14, fontweight="bold")
ax.set_ylim(-1.1, len(drugs) - 0.4)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
title(ax, R2_TITLE)
caption(fig, "Bar = average SHAP contribution of the drug itself (log-odds scale), after side effects, reporter type, age, sex, "
        "reason for use and the other inputs are accounted for. Under each name: raw share of the drug's reports that are serious. "
        "Linked to, not caused by. " + SOURCE)
fig.savefig(FIG_DIR / "R2_drug_effect.png")
plt.show()
pd.DataFrame({"drug's own push (SHAP)": drug_push.round(3), "raw % serious": raw_pct.round(1)}).loc[drugs[::-1]]

## R3: If reviewers read the riskiest reports first, how many serious ones do they find?

The model gives every test report a risk score. We sort the reports from highest to lowest score
and cut them into 10 equal bands. If the model is useful, the top bands hold far more serious reports
than the 11% average. This is how the model would actually be used: a **to-do list for reviewers**,
like a triage nurse deciding who is seen first.

In [ ]:
bands = pd.DataFrame({"prob": prob, "serious": y_test.values})
bands["band"] = pd.qcut(bands["prob"].rank(method="first", ascending=False), 10, labels=range(1, 11))
band_stats = bands.groupby("band", observed=True)["serious"].agg(["mean", "sum", "size"])
band_stats["pct_serious"] = band_stats["mean"] * 100
band_stats["pct_of_all_serious"] = band_stats["sum"] / band_stats["sum"].sum() * 100
overall = y_test.mean() * 100
top_band = band_stats.iloc[0]
top2_found = band_stats["pct_of_all_serious"].iloc[:2].sum()

fig, ax = plt.subplots(figsize=(12, 6.75))
labels = ["1\nriskiest"] + [str(i) for i in range(2, 10)] + ["10\nleast risky"]
colors = [HIGHLIGHT] + [GREY] * 9
bars = ax.bar(labels, band_stats["pct_serious"], color=colors, width=0.7)
for b, v in zip(bars, band_stats["pct_serious"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.0f}%", ha="center", fontsize=13)
ax.axhline(overall, color="#333333", ls="--", lw=1.2)
ax.text(9.45, overall + 1.2, f"average: {overall:.0f}%", ha="right", fontsize=13)
ax.set_ylabel("% of reports in the group that are serious")
ax.set_xlabel("Risk group: reports sorted by the model's risk score into 10 equal groups")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(0, max(band_stats["pct_serious"]) * 1.15)
R3_TITLE = f"Reading the riskiest 10% of reports first finds {top_band['pct_of_all_serious']:.0f}% of all serious ones"
title(ax, R3_TITLE)
caption(fig, f"{len(y_test):,} test reports. The riskiest 10% are {top_band['pct_serious']:.0f}% serious, versus {overall:.0f}% "
        f"on average; the riskiest 20% contain {top2_found:.0f}% of all serious reports. " + SOURCE)
fig.savefig(FIG_DIR / "R3_risk_bands.png")
plt.show()
band_stats[["size", "sum", "pct_serious", "pct_of_all_serious"]].round(1)

## Step 4: Save the results for the dashboards and the report

These files let the Streamlit and Power BI dashboards show the same model results:

| File | One row per |
|---|---|
| `outputs/ml/predictions.csv` | Test report (with drug, age, sex, reporter, reason, the true answer and the risk score) |
| `outputs/ml/metrics.csv` | Model |
| `outputs/ml/feature_importance.csv` | Model input (plain-English name + SHAP importance) |

In [ ]:
pred_out = pd.concat([ids_test.reset_index(drop=True),
                      raw_test[["drug", "age_group", "sex", "reporter_type", "indication_group", "year"]].reset_index(drop=True)],
                     axis=1)
pred_out["y_true"] = y_test.values
pred_out["y_prob"] = prob.round(4)
pred_out["y_pred"] = pred
pred_out["model"] = "XGBoost"
pred_out.to_csv(ML_DIR / "predictions.csv", index=False)

metrics.to_csv(ML_DIR / "metrics.csv", index=False)

fi = importance.reset_index().rename(columns={"index": "column"})
fi["share_pct"] = (fi["mean_abs_shap"] / fi["mean_abs_shap"].sum() * 100).round(2)
fi.insert(0, "model", "XGBoost")
fi.to_csv(ML_DIR / "feature_importance.csv", index=False)

for f in ["predictions.csv", "metrics.csv", "feature_importance.csv"]:
    print("saved ->", ML_DIR / f)
print("saved -> figures/R1_top_factors.png, R2_drug_effect.png, R3_risk_bands.png")

## One-line takeaways (for the slides and the report)

In [ ]:
t = top.iloc[::-1]
print("R1 raises:", list(t.loc[t["push"] > 0, "label"]))
print("R1 lowers:", list(t.loc[t["push"] < 0, "label"]))
print("R2 drug push (high to low):", {d: round(drug_push[d], 2) for d in drugs[::-1]})
print("R3:", f"The riskiest 10% of reports are {top_band['pct_serious']:.0f}% serious (vs {overall:.0f}% average) and hold "
      f"{top_band['pct_of_all_serious']:.0f}% of all serious reports; the riskiest 20% hold {top2_found:.0f}%.")
print("Model:", metrics.drop(columns=["model", "threshold"]).iloc[0].to_dict())